<a href="https://colab.research.google.com/github/ayeshayaz/python-practice/blob/main/ibis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%pip install 'ibis-framework[polars,duckdb,pandas]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 17.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: packaging
    Found existing installation: packaging 26.2
    Uninstalling packaging-26.2:
      Successfully uninstalled packaging-26.2
  Attempting uninstall: duckdb
    Found existing installation: duckdb 1.3.2
    Uninstalling duckdb-1.3.2:
      Successfully uninstalled duckdb-1.3.2


In [1]:
! wget https://calmcode.io/static/data/birthdays.csv


--2026-07-16 10:40:59--  https://calmcode.io/static/data/birthdays.csv
Resolving calmcode.io (calmcode.io)... 172.66.0.96, 162.159.140.98, 2606:4700:7::60, ...
Connecting to calmcode.io (calmcode.io)|172.66.0.96|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11922717 (11M) [application/octet-stream]
Saving to: ‘birthdays.csv’

birthdays.csv       100%[===================>]  11.37M  --.-KB/s    in 0.1s    

2026-07-16 10:40:59 (117 MB/s) - ‘birthdays.csv’ saved [11922717/11922717]



In [2]:
import ibis

ibis.options.interactive = True

In [3]:
con_polars = ibis.polars.connect()
tbl_polars = con_polars.read_csv("birthdays.csv")

con_duckdb = ibis.duckdb.connect()
tbl_duckdb = con_duckdb.read_csv("birthdays.csv")


In [4]:
def set_types(dataf):
    return dataf.mutate(dataf.date.to_date("%Y-%m-%d").name('date'))

def counter(dataf, *args):
    return (
        dataf
         .group_by(args)
         .agg(
             dataf.births.sum().name('sum'),
             dataf.births.mean().name('mean')
         ).order_by(args)
    )

counter(tbl_duckdb, 'date')
counter(tbl_polars, 'date')


┏━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┓
┃ date       ┃ sum   ┃ mean       ┃
┡━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━┩
│ string     │ int64 │ float64    │
├────────────┼───────┼────────────┤
│ 1969-01-01 │  8486 │ 166.392157 │
│ 1969-01-02 │  9002 │ 176.509804 │
│ 1969-01-03 │  9542 │ 187.098039 │
│ 1969-01-04 │  8960 │ 175.686275 │
│ 1969-01-05 │  8390 │ 164.509804 │
│ 1969-01-06 │  9560 │ 187.450980 │
│ 1969-01-07 │  9738 │ 190.941176 │
│ 1969-01-08 │  9734 │ 190.862745 │
│ 1969-01-09 │  9434 │ 184.980392 │
│ 1969-01-10 │ 10042 │ 196.901961 │
│ …          │     … │          … │
└────────────┴───────┴────────────┘

In [5]:
def set_types(dataf):
    return dataf.mutate(dataf.date.to_date("%Y-%m-%d").name('date'))

def counter(dataf, *args):
    return (
        dataf
         .group_by(args)
         .agg(
             dataf.births.sum().name('sum'),
             dataf.births.mean().name('mean')
         ).order_by(args)
    )

tbl_polars.pipe(set_types).pipe(counter, 'date')

/tmp/ipykernel_2988/3654090160.py:2: FutureWarning: `StringValue.to_date` is deprecated as of v10.0; use as_date() instead
  return dataf.mutate(dataf.date.to_date("%Y-%m-%d").name('date'))


┏━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┓
┃ date       ┃ sum   ┃ mean       ┃
┡━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━┩
│ date       │ int64 │ float64    │
├────────────┼───────┼────────────┤
│ 1969-01-01 │  8486 │ 166.392157 │
│ 1969-01-02 │  9002 │ 176.509804 │
│ 1969-01-03 │  9542 │ 187.098039 │
│ 1969-01-04 │  8960 │ 175.686275 │
│ 1969-01-05 │  8390 │ 164.509804 │
│ 1969-01-06 │  9560 │ 187.450980 │
│ 1969-01-07 │  9738 │ 190.941176 │
│ 1969-01-08 │  9734 │ 190.862745 │
│ 1969-01-09 │  9434 │ 184.980392 │
│ 1969-01-10 │ 10042 │ 196.901961 │
│ …          │     … │          … │
└────────────┴───────┴────────────┘

In [6]:
tbl_duckdb.pipe(counter, 'date')


┏━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┓
┃ date       ┃ sum   ┃ mean       ┃
┡━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━┩
│ date       │ int64 │ float64    │
├────────────┼───────┼────────────┤
│ 1969-01-01 │  8486 │ 166.392157 │
│ 1969-01-02 │  9002 │ 176.509804 │
│ 1969-01-03 │  9542 │ 187.098039 │
│ 1969-01-04 │  8960 │ 175.686275 │
│ 1969-01-05 │  8390 │ 164.509804 │
│ 1969-01-06 │  9560 │ 187.450980 │
│ 1969-01-07 │  9738 │ 190.941176 │
│ 1969-01-08 │  9734 │ 190.862745 │
│ 1969-01-09 │  9434 │ 184.980392 │
│ 1969-01-10 │ 10042 │ 196.901961 │
│ …          │     … │          … │
└────────────┴───────┴────────────┘

In [7]:
tbl_duckdb.pipe(counter, 'date', 'state')


┏━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━┓
┃ date       ┃ state  ┃ sum   ┃ mean    ┃
┡━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━┩
│ date       │ string │ int64 │ float64 │
├────────────┼────────┼───────┼─────────┤
│ 1969-01-01 │ AK     │    14 │    14.0 │
│ 1969-01-01 │ AL     │   174 │   174.0 │
│ 1969-01-01 │ AR     │    78 │    78.0 │
│ 1969-01-01 │ AZ     │    84 │    84.0 │
│ 1969-01-01 │ CA     │   824 │   824.0 │
│ 1969-01-01 │ CO     │   100 │   100.0 │
│ 1969-01-01 │ CT     │    90 │    90.0 │
│ 1969-01-01 │ DC     │    88 │    88.0 │
│ 1969-01-01 │ DE     │    32 │    32.0 │
│ 1969-01-01 │ FL     │   288 │   288.0 │
│ …          │ …      │     … │       … │
└────────────┴────────┴───────┴─────────┘

In [8]:
ibis.to_sql(tbl_duckdb.pipe(counter, 'date', 'state'))


```sql
SELECT
  *
FROM (
  SELECT
    "t0"."date",
    "t0"."state",
    SUM("t0"."births") AS "sum",
    AVG("t0"."births") AS "mean"
  FROM "ibis_read_csv_skwdncir5rdxvpywbelpfimolu" AS "t0"
  GROUP BY
    1,
    2
) AS "t1"
ORDER BY
  "t1"."date" ASC,
  "t1"."state" ASC
```

In [9]:
SELECT
  *
FROM (
  SELECT
    "t0"."date",
    "t0"."state",
    SUM("t0"."births") AS "sum",
    AVG("t0"."births") AS "mean"
  FROM "ibis_read_csv_qqgyzooif5exlpgcmykkyzk2ga" AS "t0"
  GROUP BY
    1,
    2
) AS "t1"
ORDER BY
  "t1"."date" ASC,
  "t1"."state" ASC

IndentationError: unexpected indent (1758402873.py, line 2)

In [10]:
sql_statement = """
SELECT
  *
FROM (
  SELECT
    "t0"."date",
    "t0"."state",
    SUM("t0"."births") AS "sum",
    AVG("t0"."births") AS "mean"
  FROM "tbl" AS "t0"
  GROUP BY
    1,
    2
) AS "t1"
ORDER BY
  "t1"."date" ASC,
  "t1"."state" ASC
"""

tbl_duckdb.alias("tbl").sql(sql_statement)

┏━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ date       ┃ state  ┃ sum            ┃ mean    ┃
┡━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ date       │ string │ decimal(38, 0) │ float64 │
├────────────┼────────┼────────────────┼─────────┤
│ 1969-01-01 │ AK     │             14 │    14.0 │
│ 1969-01-01 │ AL     │            174 │   174.0 │
│ 1969-01-01 │ AR     │             78 │    78.0 │
│ 1969-01-01 │ AZ     │             84 │    84.0 │
│ 1969-01-01 │ CA     │            824 │   824.0 │
│ 1969-01-01 │ CO     │            100 │   100.0 │
│ 1969-01-01 │ CT     │             90 │    90.0 │
│ 1969-01-01 │ DC     │             88 │    88.0 │
│ 1969-01-01 │ DE     │             32 │    32.0 │
│ 1969-01-01 │ FL     │            288 │   288.0 │
│ …          │ …      │              … │       … │
└────────────┴────────┴────────────────┴─────────┘